# Feature Engineering and Target Creation

## Project
Explainable Machine Learning for Chronic Kidney Disease Prediction

## Objectives
- Understand patient follow-up measurements
- Identify patients already below the progression threshold
- Investigate future kidney-function decline
- Define a suitable prediction target
- Construct patient-level features
- Prepare the data for machine learning

In [8]:

import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path(
    r"C:\Users\DELL\Desktop\CRONIC KIDNEY DISEASES"
)

data_path = project_dir / "datasets" / "processed" / "smart_c_merged.csv"

df = pd.read_csv(data_path)

print("Dataset Shape:", df.shape)
print("Unique Patients:", df["patient_id"].nunique())

display(df.head())


Dataset Shape: (41800, 15)
Unique Patients: 5000


,patient_id,randomized_flag,itt_flag,treatment_flag,analysis_flag,egfr_value,parameter_code,parameter_name,study_day,visit_number,visit_label,treatment_group,baseline_egfr,baseline_glp1,egfr_category
0,id0001,Y,Y,Y,Y,54,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,1,0,BASELINE,0,54,N,Screening eGFR 45 to <60 mL/min/1.73m2
1,id0001,Y,Y,Y,Y,51,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,22,3,WEEK 3,0,54,N,Screening eGFR 45 to <60 mL/min/1.73m2
2,id0001,Y,Y,Y,Y,61,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,91,13,WEEK 13,0,54,N,Screening eGFR 45 to <60 mL/min/1.73m2
3,id0001,Y,Y,Y,Y,63,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,183,26,WEEK 26,0,54,N,Screening eGFR 45 to <60 mL/min/1.73m2
4,id0001,Y,Y,Y,Y,53,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,362,52,WEEK 52,0,54,N,Screening eGFR 45 to <60 mL/min/1.73m2


In [9]:


# Select patients with baseline eGFR below 30
low_baseline = df[
    df["baseline_egfr"] < 30
]

# Count unique patients
low_baseline_patients = low_baseline["patient_id"].nunique()

# Count all patients
total_patients = df["patient_id"].nunique()

# Patients with baseline eGFR >= 30
eligible_patients = df[
    df["baseline_egfr"] >= 30
]["patient_id"].nunique()

print("Total Patients:", total_patients)

print("Patients with baseline eGFR < 30:",
      low_baseline_patients)

print("Patients with baseline eGFR >= 30:",
      eligible_patients)

# Calculate percentage
percentage = (low_baseline_patients / total_patients) * 100

print("Percentage with baseline eGFR < 30:",
      round(percentage, 2), "%")

# Display example patients
display(
    low_baseline[
        ["patient_id", "baseline_egfr"]
    ].drop_duplicates().head(10)
)


Total Patients: 5000
Patients with baseline eGFR < 30: 284
Patients with baseline eGFR >= 30: 4716
Percentage with baseline eGFR < 30: 5.68 %


,patient_id,baseline_egfr
9,id0002,27
302,id0038,29
398,id0050,24
491,id0061,24
987,id0119,27
1012,id0122,18
1192,id0144,27
1366,id0165,23
1523,id0184,29
1808,id0218,25


In [10]:



# Select measurements after the baseline visit
future_records = df[
    df["visit_number"] > 0
]

# Find measurements below 30
low_future_egfr = future_records[
    future_records["egfr_value"] < 30
]

# Count unique patients
future_low_patients = low_future_egfr[
    "patient_id"
].nunique()

print("Patients with any follow-up eGFR < 30:",
      future_low_patients)

# Only consider patients whose baseline eGFR was >= 30
new_low_egfr = low_future_egfr[
    low_future_egfr["baseline_egfr"] >= 30
]

new_low_patients = new_low_egfr[
    "patient_id"
].nunique()

print("Patients crossing below 30 after baseline:",
      new_low_patients)

# Display example records
display(
    new_low_egfr[
        [
            "patient_id",
            "baseline_egfr",
            "egfr_value",
            "study_day",
            "visit_label",
            "analysis_flag"
        ]
    ].head(10)
)


Patients with any follow-up eGFR < 30: 130
Patients crossing below 30 after baseline: 123


,patient_id,baseline_egfr,egfr_value,study_day,visit_label,analysis_flag
180,id0022,42,28,543,WEEK 78,Y
412,id0051,58,24,910,WEEK 130,Y
804,id0098,90,26,22,WEEK 3,Y
1270,id0153,35,29,364,WEEK 52,Y
2885,id0347,73,15,728,WEEK 104,Y
3054,id0367,38,28,727,WEEK 104,Y
3104,id0373,36,28,909,WEEK 130,Y
3172,id0381,38,28,1091,WEEK 156,Y
3256,id0391,50,22,910,WEEK 130,Y
3524,id0423,32,29,1092,WEEK 156,Y


In [11]:


# Select only follow-up visits
followup_records = df[
    df["visit_number"] > 0
]

# Find latest study day for each patient
patient_followup = followup_records.groupby(
    "patient_id"
)["study_day"].max()

print("Latest follow-up day for each patient:")
display(patient_followup.head(10))

# Summary statistics
print("\nFollow-up Duration Summary:")
display(patient_followup.describe())

# Find shortest and longest recorded follow-up
print("Minimum latest follow-up day:",
      patient_followup.min())

print("Maximum latest follow-up day:",
      patient_followup.max())

# Count patients with measurements at or beyond day 365
one_year_patients = (
    patient_followup >= 365
).sum()

print("Patients with a measurement at or after day 365:",
      one_year_patients)


Latest follow-up day for each patient:


patient_id
id0001    1092
id0002    1092
id0003    1091
id0004     911
id0005    1092
id0006    1092
id0007     911
id0008    1092
id0009    1091
id0010     911
Name: study_day, dtype: int64


Follow-up Duration Summary:


count    5000.000000
mean     1044.802000
std        98.743948
min       365.000000
25%      1090.000000
50%      1091.000000
75%      1092.000000
max      1096.000000
Name: study_day, dtype: float64

Minimum latest follow-up day: 365
Maximum latest follow-up day: 1096
Patients with a measurement at or after day 365: 5000


In [ ]:


visit_summary = df.groupby(
    ["visit_number", "visit_label"]
).agg(
    total_records=("patient_id", "size"),
    unique_patients=("patient_id", "nunique"),
    earliest_day=("study_day", "min"),
    latest_day=("study_day", "max")
)

print("Follow-up Visit Summary:")
display(visit_summary)


Follow-up Visit Summary:


,,total_records,unique_patients,earliest_day,latest_day
visit_number,visit_label,,,,
0,BASELINE,5000,5000,1,1
3,WEEK 3,5085,4980,15,24
13,WEEK 13,4986,4893,85,94
26,WEEK 26,4905,4810,177,185
52,WEEK 52,4727,4634,358,367
78,WEEK 78,4497,4414,541,549
104,WEEK 104,4358,4272,722,731
130,WEEK 130,4219,4131,905,913
156,WEEK 156,4023,3943,1087,1096


In [ ]:


flag_summary = pd.crosstab(
    df["visit_label"],
    df["analysis_flag"].fillna("Missing")
)

display(flag_summary)

# Count patients with a selected Week 52 measurement
week52 = df[
    (df["visit_number"] == 52) &
    (df["analysis_flag"] == "Y")
]

print("Patients with selected Week 52 measurement:",
      week52["patient_id"].nunique())


analysis_flag,Missing,Y
visit_label,,
BASELINE,0,5000
WEEK 104,97,4261
WEEK 13,94,4892
WEEK 130,109,4110
WEEK 156,102,3921
WEEK 26,99,4806
WEEK 3,105,4980
WEEK 52,102,4625
WEEK 78,92,4405


Patients with selected Week 52 measurement: 4625


In [14]:


# Step 1: Select eligible patients
eligible = df[df["baseline_egfr"] >= 30].copy()

# Step 2: Select analysis-flagged follow-up measurements
followup_selected = eligible[
    (eligible["visit_number"] > 0) &
    (eligible["visit_number"] <= 156) &
    (eligible["analysis_flag"] == "Y")
]

# Step 3: Identify patients crossing below eGFR 30
progression_records = followup_selected[
    followup_selected["egfr_value"] < 30
]

# Step 4: Count unique patients
progression_patients = progression_records["patient_id"].nunique()

print("Eligible patients:", eligible["patient_id"].nunique())

print("Patients crossing below 30 within 3 years:",
      progression_patients)

# Step 5: Display example progression records
display(
    progression_records[
        ["patient_id", "baseline_egfr",
         "egfr_value", "study_day", "visit_label"]
    ].head(10)
)


Eligible patients: 4716
Patients crossing below 30 within 3 years: 123


,patient_id,baseline_egfr,egfr_value,study_day,visit_label
180,id0022,42,28,543,WEEK 78
412,id0051,58,24,910,WEEK 130
804,id0098,90,26,22,WEEK 3
1270,id0153,35,29,364,WEEK 52
2885,id0347,73,15,728,WEEK 104
3054,id0367,38,28,727,WEEK 104
3104,id0373,36,28,909,WEEK 130
3172,id0381,38,28,1091,WEEK 156
3256,id0391,50,22,910,WEEK 130
3524,id0423,32,29,1092,WEEK 156


In [15]:


# One row per eligible patient
eligible_ids = eligible["patient_id"].drop_duplicates()

# Patients with selected Week 156 measurements
final_visit_ids = followup_selected[
    followup_selected["visit_number"] == 156
]["patient_id"].unique()

# Patients with an observed threshold crossing
event_ids = progression_records["patient_id"].unique()

# Patients without a recorded progression event
no_event_ids = eligible_ids[
    ~eligible_ids.isin(event_ids)
]

# Among patients without events, identify those
# with a selected final visit
complete_no_event = no_event_ids[
    no_event_ids.isin(final_visit_ids)
]

# Patients without events and without final visit data
incomplete_no_event = no_event_ids[
    ~no_event_ids.isin(final_visit_ids)
]

print("Eligible patients:", len(eligible_ids))
print("Observed event patients:", len(event_ids))
print("No-event patients with final visit:",
      len(complete_no_event))
print("No-event patients without final visit:",
      len(incomplete_no_event))


Eligible patients: 4716
Observed event patients: 123
No-event patients with final visit: 3594
No-event patients without final visit: 999


In [16]:

# Cell 10: Create patient-level progression labels

# Create patient-level baseline information
patient_data = eligible[
    ["patient_id", "baseline_egfr",
     "treatment_group", "baseline_glp1",
     "egfr_category"]
].drop_duplicates(subset=["patient_id"]).copy()

# Initially mark targets as unknown
patient_data["ckd_progression"] = pd.NA

# Assign 1 to patients with an observed progression event
patient_data.loc[
    patient_data["patient_id"].isin(event_ids),
    "ckd_progression"
] = 1

# Assign 0 to complete no-event patients
patient_data.loc[
    patient_data["patient_id"].isin(complete_no_event),
    "ckd_progression"
] = 0

# Check label distribution including unknown outcomes
print("Target Distribution:")
print(
    patient_data["ckd_progression"].value_counts(
        dropna=False
    )
)

display(patient_data.head(10))


Target Distribution:
ckd_progression
0       3594
<NA>     999
1        123
Name: count, dtype: int64


,patient_id,baseline_egfr,treatment_group,baseline_glp1,egfr_category,ckd_progression
0,id0001,54,0,N,Screening eGFR 45 to <60 mL/min/1.73m2,0
16,id0003,50,1,N,Screening eGFR 45 to <60 mL/min/1.73m2,0
24,id0004,47,0,N,Screening eGFR 45 to <60 mL/min/1.73m2,<NA>
33,id0005,75,0,N,Screening eGFR 60 to <90 mL/min/1.73m2,0
42,id0006,30,0,N,Screening eGFR 30 to <45 mL/min/1.73m2,0
50,id0007,56,0,N,Screening eGFR 45 to <60 mL/min/1.73m2,<NA>
58,id0008,59,1,N,Screening eGFR 60 to <90 mL/min/1.73m2,0
67,id0009,65,1,N,Screening eGFR 60 to <90 mL/min/1.73m2,0
75,id0010,70,1,N,Screening eGFR 60 to <90 mL/min/1.73m2,<NA>
82,id0011,45,0,N,Screening eGFR 30 to <45 mL/min/1.73m2,<NA>


In [17]:

# Keep patients with known provisional outcomes
model_data = patient_data.dropna(
    subset=["ckd_progression"]
).copy()

# Convert target to integer
model_data["ckd_progression"] = (
    model_data["ckd_progression"].astype(int)
)

print("Model Dataset Shape:", model_data.shape)

print("\nTarget Distribution:")
print(model_data["ckd_progression"].value_counts())

print("\nMissing Values:")
print(model_data.isnull().sum())

display(model_data.head())


Model Dataset Shape: (3717, 6)

Target Distribution:
ckd_progression
0    3594
1     123
Name: count, dtype: int64

Missing Values:
patient_id         0
baseline_egfr      0
treatment_group    0
baseline_glp1      0
egfr_category      0
ckd_progression    0
dtype: int64


,patient_id,baseline_egfr,treatment_group,baseline_glp1,egfr_category,ckd_progression
0,id0001,54,0,N,Screening eGFR 45 to <60 mL/min/1.73m2,0
16,id0003,50,1,N,Screening eGFR 45 to <60 mL/min/1.73m2,0
33,id0005,75,0,N,Screening eGFR 60 to <90 mL/min/1.73m2,0
42,id0006,30,0,N,Screening eGFR 30 to <45 mL/min/1.73m2,0
58,id0008,59,1,N,Screening eGFR 60 to <90 mL/min/1.73m2,0


In [18]:

# Cell 12: Count progression-related measurements

# Number of eGFR values below 30 per patient
low_measurement_counts = progression_records.groupby(
    "patient_id"
).size()

print("Number of low eGFR measurements per patient:")
display(low_measurement_counts.head(10))

# Count patients with one low measurement
single_low = (low_measurement_counts == 1).sum()

# Count patients with at least two low measurements
multiple_low = (low_measurement_counts >= 2).sum()

print("\nPatients with exactly one low measurement:", single_low)
print("Patients with two or more low measurements:", multiple_low)

print("\nTotal progression candidates:",
      low_measurement_counts.size)


Number of low eGFR measurements per patient:


patient_id
id0022    1
id0051    1
id0098    1
id0153    1
id0347    1
id0367    1
id0373    1
id0381    1
id0391    1
id0423    1
dtype: int64


Patients with exactly one low measurement: 121
Patients with two or more low measurements: 2

Total progression candidates: 123


In [19]:

# Cell 13: Investigate timing of low eGFR measurements

low_timing = progression_records.groupby(
    "patient_id"
).agg(
    low_measurements=("egfr_value", "size"),
    first_low_day=("study_day", "min"),
    last_low_day=("study_day", "max"),
    minimum_egfr=("egfr_value", "min")
)

# Calculate duration between first and last low measurement
low_timing["days_between_low"] = (
    low_timing["last_low_day"] -
    low_timing["first_low_day"]
)

print("Low eGFR timing summary:")
display(low_timing.head(10))

print("\nPatients with low measurements >= 90 days apart:")
print((low_timing["days_between_low"] >= 90).sum())


Low eGFR timing summary:


,low_measurements,first_low_day,last_low_day,minimum_egfr,days_between_low
patient_id,,,,,
id0022,1,543,543,28,0
id0051,1,910,910,24,0
id0098,1,22,22,26,0
id0153,1,364,364,29,0
id0347,1,728,728,15,0
id0367,1,727,727,28,0
id0373,1,909,909,28,0
id0381,1,1091,1091,28,0
id0391,1,910,910,22,0



Patients with low measurements >= 90 days apart:
2


In [20]:


decline_data = followup_selected.copy()

# Calculate percentage decline
decline_data["egfr_decline_pct"] = (
    (decline_data["baseline_egfr"] - decline_data["egfr_value"])
    / decline_data["baseline_egfr"]
) * 100

# Find measurements with at least 40% decline
decline_40 = decline_data[
    decline_data["egfr_decline_pct"] >= 40
]

print("Patients with at least one 40% eGFR decline:",
      decline_40["patient_id"].nunique())

display(
    decline_40[
        ["patient_id", "baseline_egfr", "egfr_value",
         "egfr_decline_pct", "study_day"]
    ].head(10)
)


Patients with at least one 40% eGFR decline: 1449


,patient_id,baseline_egfr,egfr_value,egfr_decline_pct,study_day
36,id0005,75,43,42.666667,181
38,id0005,75,42,44.000000,544
39,id0005,75,33,56.000000,726
40,id0005,75,32,57.333333,909
69,id0009,65,37,43.076923,89
90,id0012,98,50,48.979592,20
92,id0012,98,56,42.857143,90
93,id0012,98,47,52.040816,179
94,id0012,98,45,54.081633,364
95,id0012,98,35,64.285714,545


In [21]:

# Cell 15: Compare progression definitions

absolute_event_ids = set(
    progression_records["patient_id"].unique()
)

relative_event_ids = set(
    decline_40["patient_id"].unique()
)

print("Absolute threshold events:",
      len(absolute_event_ids))

print("40% decline events:",
      len(relative_event_ids))

print("Patients meeting both definitions:",
      len(absolute_event_ids & relative_event_ids))

print("Only absolute threshold:",
      len(absolute_event_ids - relative_event_ids))

print("Only 40% decline:",
      len(relative_event_ids - absolute_event_ids))


Absolute threshold events: 123
40% decline events: 1449
Patients meeting both definitions: 80
Only absolute threshold: 43
Only 40% decline: 1369
